## Instalación de dependencias

In [1]:
%pip install matplotlib numpy plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 22.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.


## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import plotly.graph_objects as go
from IPython.display import display
import time
import os

os.makedirs('exports', exist_ok=True)


## Fuente de datos: simulación de 4 señales

Se simulan cuatro métricas independientes que representan escenarios reales de monitoreo:
- **Temperatura**: onda sinusoidal con ruido gaussiano
- **Pulso cardíaco**: onda sinusoidal rápida con picos aleatorios
- **Conteo de objetos**: entero aleatorio entre 0 y 20 (simula detección YOLO)
- **Señal de posición Y**: coseno desplazado (simula coordenada de nariz en MediaPipe)

In [2]:
N_STEPS   = 200 
t         = np.linspace(0, 4 * np.pi, N_STEPS)

#Temperatura simulada (°C)
temp = 22 + 3 * np.sin(t) + np.random.normal(0, 0.4, N_STEPS)

#Pulso cardíaco (bpm)
pulso = 75 + 10 * np.sin(3 * t) + np.random.normal(0, 1.5, N_STEPS)

#Conteo de objetos (simula salida YOLO)
conteo = np.random.randint(0, 20, N_STEPS).astype(float)
conteo = pd.Series(conteo).rolling(window=5, min_periods=1).mean().values

#Coordenada Y normalizada (simula nariz en MediaPipe)
pos_y = 0.5 + 0.3 * np.cos(t * 1.5) + np.random.normal(0, 0.02, N_STEPS)
pos_y = np.clip(pos_y, 0, 1)

print(f' {N_STEPS} frames de datos simulados')
print(f'   Temperatura  → min {temp.min():.1f}°C  max {temp.max():.1f}°C')
print(f'   Pulso        → min {pulso.min():.1f}   max {pulso.max():.1f} bpm')
print(f'   Conteo       → min {conteo.min():.1f}  max {conteo.max():.1f} obj')
print(f'   Posición Y   → min {pos_y.min():.3f}  max {pos_y.max():.3f}')

 200 frames de datos simulados
   Temperatura  → min 18.5°C  max 26.0°C
   Pulso        → min 63.0   max 87.3 bpm
   Conteo       → min 1.8  max 15.6 obj
   Posición Y   → min 0.170  max 0.831


## Visualización 1: Temperatura en tiempo real con FuncAnimation

`FuncAnimation` actualiza la gráfica frame por frame simulando una fuente de datos en vivo, y es importante setearla en 50 frames, que evita que la línea crezca indefinidamente.

In [4]:
%matplotlib tk

WINDOW = 50  

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

line, = ax.plot([], [], color='#e94560', linewidth=2, label='Temperatura (°C)')
ax.set_xlim(0, WINDOW)
ax.set_ylim(temp.min() - 1, temp.max() + 1)
ax.set_xlabel('Frame', color='white')
ax.set_ylabel('°C', color='white')
ax.set_title('Temperatura en Tiempo Real', color='white', fontsize=13)
ax.tick_params(colors='white')
ax.legend(facecolor='#0f3460', labelcolor='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

text_val = ax.text(0.02, 0.92, '', transform=ax.transAxes,
                   color='#e94560', fontsize=11, fontweight='bold')

x_data, y_data = [], []

def init():
    line.set_data([], [])
    return line,

def update(frame):
    x_data.append(frame)
    y_data.append(temp[frame])

    x_win = x_data[-WINDOW:]
    y_win = y_data[-WINDOW:]

    x_rel = list(range(len(x_win)))
    line.set_data(x_rel, y_win)
    text_val.set_text(f'{temp[frame]:.1f} °C')
    return line, text_val

ani = animation.FuncAnimation(
    fig, update, frames=N_STEPS,
    init_func=init, interval=40, blit=True, repeat=False
)

plt.tight_layout()
plt.show()

## Visualización 2: Dashboard de 4 señales simultáneas con FuncAnimation

Las cuatro señales se muestran en subplots sincronizados, actualizando todas en el mismo frame.

In [ ]:
%matplotlib tk

WINDOW = 60

fig, axes = plt.subplots(4, 1, figsize=(11, 9))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Dashboard Tiempo Real — 4 Señales', color='white', fontsize=14, y=0.98)

configs = [
    (temp,   'Temperatura',   '°C',   '#e94560'),
    (pulso,  'Pulso cardíaco','bpm',  '#0f3460'),
    (conteo, 'Conteo objetos','obj',  '#533483'),
    (pos_y,  'Posición Y',    'norm', '#e8a838'),
]

lines_db = []
texts_db = []
buffers  = [{'x': [], 'y': []} for _ in configs]

for ax, (data, label, unit, color) in zip(axes, configs):
    ax.set_facecolor('#16213e')
    ln, = ax.plot([], [], color=color, linewidth=1.8)
    ax.set_xlim(0, WINDOW)
    ax.set_ylim(data.min() - abs(data.min()) * 0.1,
                data.max() + abs(data.max()) * 0.1)
    ax.set_ylabel(f'{label} ({unit})', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
    tx = ax.text(0.98, 0.85, '', transform=ax.transAxes,
                 color=color, fontsize=9, ha='right', fontweight='bold')
    lines_db.append(ln)
    texts_db.append(tx)

axes[-1].set_xlabel('Frame', color='white', fontsize=9)

def update_db(frame):
    artists = []
    for i, (data, _, unit, _) in enumerate(configs):
        buffers[i]['x'].append(frame)
        buffers[i]['y'].append(data[frame])
        x_win = list(range(min(len(buffers[i]['x']), WINDOW)))
        y_win = buffers[i]['y'][-WINDOW:]
        lines_db[i].set_data(x_win, y_win)
        texts_db[i].set_text(f'{data[frame]:.2f} {unit}')
        artists += [lines_db[i], texts_db[i]]
    return artists

ani_db = animation.FuncAnimation(
    fig, update_db, frames=N_STEPS,
    interval=40, blit=True, repeat=False
)

plt.tight_layout()
plt.show()

✓ Dashboard de 4 señales ejecutado


## Visualización 3: Conteo de objetos con barras animadas

El gráfico de barras acumula un histograma de los últimos N valores de conteo, mostrando la distribución de frecuencias mientras la señal avanza.

In [ ]:
%matplotlib tk

HIST_WIN = 80  
N_BINS   = 10

fig_bar, ax_bar = plt.subplots(figsize=(9, 4))
fig_bar.patch.set_facecolor('#1a1a2e')
ax_bar.set_facecolor('#16213e')
ax_bar.set_title('Distribución de Conteo de Objetos (últimos 80 frames)', color='white', fontsize=12)
ax_bar.set_xlabel('Cantidad de objetos', color='white')
ax_bar.set_ylabel('Frecuencia', color='white')
ax_bar.tick_params(colors='white')
for spine in ax_bar.spines.values():
    spine.set_edgecolor('#333')

bar_buffer = []
bins_edges = np.linspace(0, 20, N_BINS + 1)
bar_centers = (bins_edges[:-1] + bins_edges[1:]) / 2
bars = ax_bar.bar(bar_centers, np.zeros(N_BINS),
                  width=1.6, color='#533483', edgecolor='#888')
ax_bar.set_ylim(0, HIST_WIN * 0.5)

frame_text = ax_bar.text(0.02, 0.92, '', transform=ax_bar.transAxes,
                          color='white', fontsize=10)

def update_bar(frame):
    bar_buffer.append(conteo[frame])
    window = bar_buffer[-HIST_WIN:]
    counts, _ = np.histogram(window, bins=bins_edges)
    for bar, h in zip(bars, counts):
        bar.set_height(h)
    frame_text.set_text(f'Frame {frame} | Valor actual: {conteo[frame]:.1f}')
    return list(bars) + [frame_text]

ani_bar = animation.FuncAnimation(
    fig_bar, update_bar, frames=N_STEPS,
    interval=40, blit=True, repeat=False
)

plt.tight_layout()
plt.show()

## Visualización 4: Gráfico interactivo estático de todas las señales (Plotly)

Plotly genera un HTML interactivo con zoom, pan y tooltips

In [ ]:
frames_idx = np.arange(N_STEPS)

fig_plotly = go.Figure()

fig_plotly.add_trace(go.Scatter(
    x=frames_idx, y=temp,
    name='Temperatura (°C)', line=dict(color='#e94560', width=2)
))
fig_plotly.add_trace(go.Scatter(
    x=frames_idx, y=pulso,
    name='Pulso (bpm)', line=dict(color='#4fc3f7', width=2)
))
fig_plotly.add_trace(go.Scatter(
    x=frames_idx, y=conteo,
    name='Conteo objetos', line=dict(color='#ce93d8', width=2)
))
fig_plotly.add_trace(go.Scatter(
    x=frames_idx, y=pos_y * 100,   # escalar para que sea visible junto a las demás
    name='Posición Y ×100', line=dict(color='#e8a838', width=2)
))

fig_plotly.update_layout(
    title='Señales Simuladas — Vista Completa (Plotly)',
    xaxis_title='Frame',
    yaxis_title='Valor',
    template='plotly_dark',
    legend=dict(orientation='h', y=1.1),
    height=480
)

fig_plotly.write_html('exports/dashboard_plotly.html')
fig_plotly.show()

## BONUS: Exportar datos a CSV y estadísticas de rendimiento

In [ ]:
df = pd.DataFrame({
    'frame':       np.arange(N_STEPS),
    'temperatura': np.round(temp,  3),
    'pulso':       np.round(pulso, 3),
    'conteo':      np.round(conteo,3),
    'pos_y':       np.round(pos_y, 4),
})

csv_path = 'exports/datos_tiempo_real.csv'
df.to_csv(csv_path, index=False)
print(df.head(8).to_string(index=False))

print('\n── Estadísticas de rendimiento ──')
fps_target = 1000 / 40  
print(f'   FPS objetivo (intervalo 40ms): {fps_target:.1f}')
print(f'   Frames totales:                {N_STEPS}')
print(f'   Duración estimada:             {N_STEPS / fps_target:.1f} segundos')

print('\n── Estadísticas por señal ──')
for col in ['temperatura', 'pulso', 'conteo', 'pos_y']:
    s = df[col]
    print(f'   {col:14s} → media {s.mean():.3f}  std {s.std():.3f}  min {s.min():.3f}  max {s.max():.3f}')

## BONUS: Guardar captura estática de las 4 señales como PNG

In [ ]:
%matplotlib inline

fig_static, axes_s = plt.subplots(4, 1, figsize=(12, 10))
fig_static.patch.set_facecolor('#1a1a2e')
fig_static.suptitle('Dashboard Señales Simuladas — Captura Completa',
                     color='white', fontsize=13, y=0.99)

plot_configs = [
    (temp,   'Temperatura',    '°C',    '#e94560'),
    (pulso,  'Pulso cardíaco', 'bpm',   '#4fc3f7'),
    (conteo, 'Conteo objetos', 'obj',   '#ce93d8'),
    (pos_y,  'Posición Y',     'norm',  '#e8a838'),
]

for ax, (data, label, unit, color) in zip(axes_s, plot_configs):
    ax.set_facecolor('#16213e')
    ax.plot(data, color=color, linewidth=1.5)
    ax.fill_between(range(len(data)), data, alpha=0.15, color=color)
    ax.set_ylabel(f'{label} ({unit})', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
    ax.axhline(data.mean(), color=color, linestyle='--', alpha=0.5, linewidth=1)
    ax.text(N_STEPS * 0.99, data.mean(),
            f'  avg {data.mean():.2f}', color=color, fontsize=8, va='center')

axes_s[-1].set_xlabel('Frame', color='white', fontsize=9)
plt.tight_layout()

png_path = 'exports/dashboard_estatico.png'
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()